# Lab 3: Spark MLlib - Machine Learning

## 🎯 **Learning Objectives:**
- Master Spark MLlib machine learning library
- Learn ML pipeline concepts and workflows
- Practice feature engineering techniques
- Understand model training and evaluation
- Implement real-world ML use cases

## 📚 **Key Concepts:**
1. **MLlib**: Spark's machine learning library
2. **ML Pipelines**: End-to-end ML workflows
3. **Feature Engineering**: Data transformation for ML
4. **Model Training**: Supervised and unsupervised learning
5. **Model Evaluation**: Performance metrics and validation

## 🏗️ **Architecture Overview:**
```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   Raw Data      │───▶│ Feature Pipeline │───▶│   ML Models     │
│   (CSV/JSON)    │    │ • Transformers   │    │ • Classification│
│                 │    │ • Estimators    │    │ • Regression    │
└─────────────────┘    └──────────────────┘    │ • Clustering   │
         │                        │               └─────────────────┘
         ▼                        ▼                        │
┌─────────────────┐    ┌──────────────────┐               ▼
│ Data Preprocessing│    │ Feature Selection│    ┌─────────────────┐
│ • Cleaning       │    │ • Scaling        │    │ Model Evaluation│
│ • Validation    │    │ • Encoding       │    │ • Metrics       │
└─────────────────┘    └──────────────────┘    │ • Validation   │
                                                └─────────────────┘
```

## 🤖 **ML Use Cases:**
- **Classification**: Customer segmentation, fraud detection
- **Regression**: Price prediction, demand forecasting
- **Clustering**: Market segmentation, anomaly detection
- **Recommendation**: Product recommendations, content filtering


In [4]:
import os
import sys

# Khởi tạo Spark Session cho Machine Learning (Cấu hình local bypass Docker errors)
try:
    from pyspark.sql import SparkSession
    from pyspark import SparkContext

    # Stop any existing SparkContext to avoid errors
    if SparkContext._active_spark_context is not None:
        SparkContext._active_spark_context.stop()

    spark = SparkSession.builder \
        .appName("Spark_ML_Practice") \
        .master("local[*]") \
        .config("spark.driver.memory", "2g") \
        .config("spark.executor.memory", "2g") \
        .getOrCreate()
        
    print("🚀 Spark ML Session initialized successfully!")
    print(f"📊 Spark Version: {spark.version}")
    print(f"🔗 Master URL: {spark.conf.get('spark.master')}")
except Exception as e:
    print(f"❌ Error initializing Spark: {e}")
    raise e

🚀 Spark ML Session initialized successfully!
📊 Spark Version: 4.1.1
🔗 Master URL: local[*]


In [5]:
# Initialize Spark Session for MLlib
from pyspark import SparkContext
SparkContext.getOrCreate().stop()

spark = SparkSession.builder \
    .appName("SparkMLlibLab") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Set log level to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

print("🚀 Spark MLlib Session initialized successfully!")
print(f"📊 Spark Version: {spark.version}")
print(f"🔗 Master URL: {spark.sparkContext.master}")
print(f"🤖 MLlib Version: {spark.version}")

🚀 Spark MLlib Session initialized successfully!
📊 Spark Version: 4.1.1
🔗 Master URL: local[*]
🤖 MLlib Version: 4.1.1


In [10]:
import builtins

print("📊 Creating sample datasets for ML experiments...")

print("\n👥 Creating customer classification dataset...")
customer_data = []
customer_types = ['Premium', 'Standard', 'Basic']
for i in range(1000):
    age = random.randint(18, 80)
    income = random.randint(20000, 150000)
    spending = random.randint(100, 10000)
    
    # Simple logic to determine churn
    churn_prob = 0.1
    if income < 40000 or spending < 500:
        churn_prob += 0.3
    if age > 60:
        churn_prob += 0.2
        
    customer_data.append({
        'customer_id': f'CUST_{i+1:04d}',
        'age': age,
        'income': income,
        'spending': spending,
        'customer_type': random.choice(customer_types),
        'churn': 1 if random.random() < churn_prob else 0
    })

print("\n💰 Creating sales regression dataset...")
sales_data = []
products = ['Electronics', 'Clothing', 'Food', 'Books']
regions = ['North', 'South', 'East', 'West']

for i in range(1000):
    marketing_budget = random.randint(1000, 50000)
    price = builtins.round(random.uniform(10.0, 1000.0), 2)
    promotion = random.choice(['None', 'TenPercent', 'BuyOneGetOne'])
    season = random.choice(['Spring', 'Summer', 'Autumn', 'Winter'])
    
    # Calculate sales based on features
    base_sales = marketing_budget * 0.5 + (1000 - price) * 2
    if promotion != 'None':
        base_sales *= 1.5
    if season == 'Winter':
        base_sales *= 1.3
        
    # Add some noise
    sales = base_sales + random.randint(-200, 200)
    # the pyspark functions namespace might override max, so use simple if/else
    sales = sales if sales > 0 else 0
    
    sales_data.append({
        'product_id': f'PROD_{i+1:04d}',
        'marketing_budget': marketing_budget,
        'price': price,
        'category': random.choice(products),
        'region': random.choice(regions),
        'promotion': promotion,
        'season': season,
        'sales': sales,
        'inventory_level': random.randint(10, 1000)
    })

# 3. Customer Clustering Dataset
print("\n🔄 Creating customer clustering dataset...")
clustering_data = []
# Create 3 distinct clusters of users based on behavior
for i in range(1000):
    cluster = random.choice([0, 1, 2])
    if cluster == 0:
        # High activity, high duration
        logins = random.randint(50, 100)
        duration = random.randint(300, 600)
        purchases = random.randint(10, 30)
    elif cluster == 1:
        # Medium activity, low duration
        logins = random.randint(10, 40)
        duration = random.randint(10, 100)
        purchases = random.randint(1, 5)
    else:
        # Low activity, medium duration
        logins = random.randint(1, 10)
        duration = random.randint(50, 200)
        purchases = 0
        
    clustering_data.append({
        'user_id': f'USER_{i+1:04d}',
        'monthly_logins': logins,
        'session_duration_mins': duration,
        'purchases_count': purchases
    })

print("\n🎯 Creating sample DataFrames...")
# Create DataFrames
df_classification = spark.createDataFrame(customer_data)
df_regression = spark.createDataFrame(sales_data)
clustering_df = spark.createDataFrame(clustering_data)

# Show sample data
print("\nClassification Data Sample:")
df_classification.show(5)

print("\nRegression Data Sample:")
df_regression.show(5)

print("\nClustering Data Sample:")
clustering_df.show(5)

📊 Creating sample datasets for ML experiments...

👥 Creating customer classification dataset...

💰 Creating sales regression dataset...

🔄 Creating customer clustering dataset...

🎯 Creating sample DataFrames...

Classification Data Sample:
+---+-----+-----------+-------------+------+--------+
|age|churn|customer_id|customer_type|income|spending|
+---+-----+-----------+-------------+------+--------+
| 21|    0|  CUST_0001|      Premium|114113|    1905|
| 41|    1|  CUST_0002|     Standard| 44944|    9422|
| 25|    0|  CUST_0003|      Premium|146845|    7422|
| 29|    0|  CUST_0004|        Basic| 50055|    5956|
| 71|    0|  CUST_0005|      Premium| 94481|    6992|
+---+-----+-----------+-------------+------+--------+
only showing top 5 rows

Regression Data Sample:
+-----------+---------------+----------------+------+----------+----------+------+------------------+------+
|   category|inventory_level|marketing_budget| price|product_id| promotion|region|             sales|season|
+-----

In [11]:
# Create Spark DataFrames for ML
print("🔄 Creating Spark DataFrames for ML experiments...")

# Convert to Spark DataFrames
customers_df = spark.createDataFrame(customer_data)
sales_df = spark.createDataFrame(sales_data)
clustering_df = spark.createDataFrame(clustering_data)

print("\n📊 Customer Classification DataFrame:")
customers_df.printSchema()
customers_df.show(5)

print("\n💰 Sales Regression DataFrame:")
sales_df.printSchema()
sales_df.show(5)

print("\n🔍 Customer Clustering DataFrame:")
clustering_df.printSchema()
clustering_df.show(5)

print(f"\n✅ ML DataFrames created successfully!")
print(f"   👥 Customers: {customers_df.count()} records")
print(f"   💰 Sales: {sales_df.count()} records")
print(f"   🔍 Clustering: {clustering_df.count()} records")


🔄 Creating Spark DataFrames for ML experiments...

📊 Customer Classification DataFrame:
root
 |-- age: long (nullable = true)
 |-- churn: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- income: long (nullable = true)
 |-- spending: long (nullable = true)

+---+-----+-----------+-------------+------+--------+
|age|churn|customer_id|customer_type|income|spending|
+---+-----+-----------+-------------+------+--------+
| 21|    0|  CUST_0001|      Premium|114113|    1905|
| 41|    1|  CUST_0002|     Standard| 44944|    9422|
| 25|    0|  CUST_0003|      Premium|146845|    7422|
| 29|    0|  CUST_0004|        Basic| 50055|    5956|
| 71|    0|  CUST_0005|      Premium| 94481|    6992|
+---+-----+-----------+-------------+------+--------+
only showing top 5 rows

💰 Sales Regression DataFrame:
root
 |-- category: string (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- marketing_budget: long (nullable = true)

## Exercise 1: Customer Classification

### 🎯 **Learning Objectives:**
- Master classification algorithms in MLlib
- Learn feature engineering for ML
- Practice ML pipeline creation
- Understand model evaluation metrics

### 📚 **Key Concepts:**
1. **Classification**: Predicting categorical outcomes
2. **Feature Engineering**: Preparing data for ML
3. **ML Pipelines**: End-to-end ML workflows
4. **Model Evaluation**: Performance metrics and validation


In [13]:
print("🎯 Exercise 1: Customer Classification")

print("\n🔧 Setting up feature engineering pipeline...")
# The dataset we defined for Customer Classification only has 'customer_type', it does not have 'region' or 'season'.
# For customer_data we have: 'age', 'income', 'spending', 'customer_type', 'churn'
categorical_cols_classification = ['customer_type']
numeric_cols_classification = ['age', 'income', 'spending']

# Create indexers for categorical columns
indexers_classification = [
    StringIndexer(inputCol=c, outputCol=f"{c}_indexed")
    for c in categorical_cols_classification
]

# Combine all features into a single vector
assembler_classification = VectorAssembler(
    inputCols=[f"{c}_indexed" for c in categorical_cols_classification] + numeric_cols_classification,
    outputCol="features"
)

print("\n📊 Building ML pipeline...")
# Create a Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(
    labelCol="churn",
    featuresCol="features",
    maxDepth=5
)

# Create the pipeline
pipeline_classification = Pipeline(
    stages=indexers_classification + [assembler_classification, dt_classifier]
)

print("\n🔄 Splitting data into train/test sets...")
train_data_classification, test_data_classification = df_classification.randomSplit([0.8, 0.2], seed=42)

print(f"   📊 Training set: {train_data_classification.count()} records")
print(f"   📊 Test set: {test_data_classification.count()} records")

print("\n🚀 Training the model...")

# Train the model
model_classification = pipeline_classification.fit(train_data_classification)
print("✅ Model training completed!")

print("\n📊 Making predictions on test set...")
predictions_classification = model_classification.transform(test_data_classification)
predictions_classification.select("customer_id", "churn", "prediction", "probability").show(5)

print("\n📈 Evaluating model performance...")
# Evaluate Area Under ROC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(predictions_classification)
print(f"   🎯 Area Under ROC: {auc:.4f}")

# Evaluate Accuracy
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="churn",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions_classification)
print(f"   🎯 Accuracy: {accuracy:.4f}")

# Get feature importance
dt_model = model_classification.stages[-1]
print("\n🔍 Feature Importance:")
feature_names = [f"{c}_indexed" for c in categorical_cols_classification] + numeric_cols_classification
importances = dt_model.featureImportances

for i, feature in enumerate(feature_names):
    print(f"   - {feature}: {importances[i]:.4f}")

🎯 Exercise 1: Customer Classification

🔧 Setting up feature engineering pipeline...

📊 Building ML pipeline...

🔄 Splitting data into train/test sets...
   📊 Training set: 802 records
   📊 Test set: 198 records

🚀 Training the model...
✅ Model training completed!

📊 Making predictions on test set...
+-----------+-----+----------+--------------------+
|customer_id|churn|prediction|         probability|
+-----------+-----+----------+--------------------+
|  CUST_0105|    0|       0.0|[0.92307692307692...|
|  CUST_0080|    1|       0.0|[0.92307692307692...|
|  CUST_0125|    0|       0.0|[0.92307692307692...|
|  CUST_0074|    0|       0.0|[0.79710144927536...|
|  CUST_0041|    0|       0.0|[0.92307692307692...|
+-----------+-----+----------+--------------------+
only showing top 5 rows

📈 Evaluating model performance...
   🎯 Area Under ROC: 0.4943
   🎯 Accuracy: 0.7576

🔍 Feature Importance:
   - customer_type_indexed: 0.0484
   - age: 0.3071
   - income: 0.3865
   - spending: 0.2580
